# Fig S10 (v2) — Variant PIP agreement with in-sample LD, ADSP + UKB meta-analysis (data processing)

Rebuilds Fig S10 from the v2 ADSP + UKB simulations (`32.Fig4A_4B_S9A.ADSP_UKB_Meta_v2`: total n = 10,000, NCP 80 / 120 / 250 / 500, one to three causal variants). The v1 notebook (`FigS10.Merged_Study_Sketch_PIP`) and its `figure_data` are left as they are.

**Question:** does fine-mapping with the estimated ADSP + UKB reference mixture give the same variant PIPs as fine-mapping with the in-sample LD? All fits are baseline SuSiE-RSS on the same meta-analysis z and variants.

| Fit | v2 name | LD |
|---|---|---|
| In-sample LD (Oracle, x axis) | `gen_insample` | LD of the sampled study rows, mixed by the realized weight |
| Estimated mixture (as Fig 4) | `manual_refpanel` | ω̂ R_ADSP + (1 − ω̂) R_UKB, ω̂ from the joint fit `mix_refpanel`, then a separate `susie_rss(z, R = mixture)` |
| True-weight mixture (comparison) | `gen_refpanel` | w R_ADSP + (1 − w) R_UKB, true sample-size weight |

The true-weight line separates the cost of estimating the weights (estimated vs true weight) from the gap between the reference panels and the in-sample LD (shared by both).

**Design choices**
- **ADSP shares 20 / 50 / 80% only.** At 0% and 100% there is no mixture to estimate, and at 100% the in-sample LD is the ADSP panel LD.
- **Balanced sample.** The same number of simulations (`n_per_setting`) in each of the 36 settings (4 NCP × 3 causal counts × 3 shares), drawn at random with a fixed seed among simulations where all three fits succeeded. A setting with fewer usable simulations is reported, not padded.
- **Per-simulation agreement, over credible set variants.** In each simulation, PIP correlation is computed over the union of the variants in a credible set of either fit (in-sample or the reference fit), not over all variants, where thousands of near-zero PIPs push the correlation to 1. It needs at least `min_cs_var` variants; otherwise it is NA, and max |ΔPIP| and the credible set counts are still reported. Simulations where neither fit reports a credible set have no CS variants and are counted separately. Correlations are never computed on PIPs pooled across simulations.
- **Scatter over all sampled simulations.** For the PIP scatter, the credible set variants of every sampled simulation are kept (union of the in-sample and estimated-mixture credible sets), so the figure shows the general pattern rather than chosen replicates.

**Inputs:** `ADSP_UKB_meta_v2/result/ncp_<NCP>/causal_<k>/prop_<share>/sample_*_meta.rds` (`[adsp_ukb_meta_v2]` in `32`)

**Outputs:**
- `figure_data/FigS10_v2.ADSP_UKB_PIP_Agreement` — one row per simulation × reference fit (estimated / true weight): PIP correlation over CS variants, number of CS variants, max |ΔPIP| over CS variants and over all variants, credible set counts, true and estimated ADSP weight
- `figure_data/FigS10_v2.ADSP_UKB_PIP_Agreement_Summary` — per setting: median and IQR of the correlation, median max |ΔPIP|, share with the same number of credible sets
- `figure_data/FigS10_v2.ADSP_UKB_PIP_CS_Variants` — in-sample, estimated-mixture and true-weight PIPs of the credible set variants of every sampled simulation (for the scatter)

In [7]:
source("../../R/utils.R")

res_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADSP_UKB_meta_v2/result"   # 32 [adsp_ukb_meta_v2]
props   <- c("0.20", "0.50", "0.80")   # ADSP shares with both cohorts present

name          <- "FigS10_v2.ADSP_UKB_PIP_Agreement"   # figure_data prefix
n_per_setting <- 100     # simulations per NCP x causal x share setting
min_cs_var    <- 5       # fewest CS variants for a PIP correlation
seed          <- 2026

ref_fits <- c(estimated = "manual_refpanel", true_weight = "gen_refpanel")   # each compared with gen_insample
ref_labels <- c(estimated = "Estimated mixture", true_weight = "True-weight mixture")

## Simulations available per setting

In [8]:
files <- tibble(file = list.files(res_dir, pattern = "^sample_.*_meta\\.rds$", recursive = TRUE, full.names = TRUE)) %>%
  mutate(ncp    = as.numeric(str_match(file, "/ncp_([0-9]+)/")[, 2]),
         causal = as.integer(str_match(file, "/causal_([0-9]+)/")[, 2]),
         prop   = str_match(file, "/prop_([0-9.]+)/")[, 2]) %>%
  filter(prop %in% props)
stopifnot(nrow(files) > 0, !anyNA(files$ncp), !anyNA(files$causal))

## output files per setting (before dropping failed or dropped simulations)
files %>% count(ncp, causal, prop) %>% pivot_wider(names_from = prop, values_from = n, names_prefix = "ADSP share ")

ncp,causal,ADSP share 0.20,ADSP share 0.50,ADSP share 0.80
<dbl>,<int>,<int>,<int>,<int>
80,1,496,496,496
80,2,479,373,469
80,3,466,344,442
120,1,496,496,496
120,2,488,385,474
120,3,474,341,418
250,1,496,496,496
250,2,483,367,457
250,3,471,318,427


## Balanced random sample and per-simulation agreement

Within each setting the files are shuffled with the fixed seed and read in that order until `n_per_setting` usable simulations (status `ok`, all three fits present) are collected, so the sample does not depend on which files happen to fail.

In [9]:
## agreement of one reference fit with the in-sample fit
agree_one <- function(ins, ref) {
  cs_ins <- unlist(ins$sets$cs); cs_ref <- unlist(ref$sets$cs)
  v <- sort(union(cs_ins, cs_ref))                       # CS variants of either fit
  a <- ins$pip[v]; b <- ref$pip[v]
  ok_cor <- length(v) >= min_cs_var && sd(a) > 0 && sd(b) > 0
  tibble(n_cs_var        = length(v),
         pip_cor         = if (ok_cor) cor(a, b) else NA_real_,
         max_abs_diff_cs = if (length(v)) max(abs(a - b)) else NA_real_,
         max_abs_diff    = max(abs(ins$pip - ref$pip)),
         n_cs_insample   = length(ins$sets$cs),
         n_cs_ref        = length(ref$sets$cs))
}

## per simulation: agreement rows, and the PIPs of its CS variants (in-sample or estimated mixture) for the scatter
read_sim <- function(f) {
  r <- tryCatch(readRDS(f), error = function(e) NULL)
  if (is.null(r) || !identical(r$status, "ok")) return(NULL)
  fits <- r$result_list
  if (any(vapply(c("gen_insample", ref_fits), function(m) is.null(fits[[m]]), logical(1)))) return(NULL)
  ins <- fits$gen_insample
  rows <- imap_dfr(ref_fits, function(m, nm) agree_one(ins, fits[[m]]) %>% mutate(comparison = nm, .before = 1))
  ## check: estimated weights refitted (manual_refpanel) vs the joint fit (mix_refpanel)
  mix <- fits$mix_refpanel; est <- fits$manual_refpanel
  rows <- rows %>% mutate(file = f, region = r$region, w = r$w, omega_ADSP = r$omega_weights[1],
                          manual_vs_joint_max_abs_diff = if (is.null(mix)) NA_real_ else max(abs(est$pip - mix$pip)),
                          manual_vs_joint_same_n_cs    = if (is.null(mix)) NA else length(mix$sets$cs) == length(est$sets$cs))
  v <- sort(union(unlist(ins$sets$cs), unlist(est$sets$cs)))
  pts <- tibble(file = f, variant = v, pip_insample = ins$pip[v], pip_estimated = est$pip[v],
                pip_true_weight = fits$gen_refpanel$pip[v], causal_variant = v %in% r$true_vars)
  list(rows = rows, pts = pts)
}

set.seed(seed)
by_setting <- files %>% group_by(ncp, causal, prop) %>% group_split()
sampled <- map(by_setting, function(s) {
  s <- s[sample(nrow(s)), ]
  out <- list(); k <- 0
  for (i in seq_len(nrow(s))) {
    x <- read_sim(s$file[i])
    if (is.null(x)) next
    k <- k + 1; out[[k]] <- x
    if (k == n_per_setting) break
  }
  tag <- function(d) mutate(d, ncp = s$ncp[1], causal = s$causal[1], prop = s$prop[1], .before = 1)
  list(rows = tag(map_dfr(out, "rows")), pts = tag(map_dfr(out, "pts")))
})
agree  <- map_dfr(sampled, "rows")
cs_pts <- map_dfr(sampled, "pts")
rm(sampled)

## simulations used per setting: all should equal n_per_setting
used <- agree %>% distinct(ncp, causal, prop, file) %>% count(ncp, causal, prop)
used %>% pivot_wider(names_from = prop, values_from = n, names_prefix = "ADSP share ")
short <- filter(used, n < n_per_setting)
if (nrow(short)) { cat("Settings with fewer than", n_per_setting, "usable simulations:\n"); print(short) }
c(settings = nrow(used), simulations = n_distinct(agree$file))

ncp,causal,ADSP share 0.20,ADSP share 0.50,ADSP share 0.80
<dbl>,<int>,<int>,<int>,<int>
80,1,100,100,100
80,2,100,100,100
80,3,100,100,100
120,1,100,100,100
120,2,100,100,100
120,3,100,100,100
250,1,100,100,100
250,2,100,100,100
250,3,100,100,100


settings simulations 
         36        3600

In [10]:
## check: manual_refpanel (weights estimated, then a separate fit on the fixed mixture) vs the joint fit
## mix_refpanel (weights estimated in the same susie_rss(R = list(...)) call)
agree %>% filter(comparison == "estimated") %>%
  summarise(n = n(), median_max_abs_diff = median(manual_vs_joint_max_abs_diff, na.rm = TRUE),
            max_max_abs_diff = max(manual_vs_joint_max_abs_diff, na.rm = TRUE),
            same_n_cs = mean(manual_vs_joint_same_n_cs, na.rm = TRUE))

## simulations without any CS variant (neither fit reports a credible set) and with too few for a correlation
agree %>% group_by(comparison) %>%
  summarise(n = n(), no_cs = sum(n_cs_var == 0), too_few_cs_var = sum(n_cs_var > 0 & is.na(pip_cor)),
            with_cor = sum(!is.na(pip_cor)), .groups = "drop")

agree <- agree %>% mutate(comparison_label = factor(ref_labels[comparison], levels = ref_labels))
save_data(select(agree, -manual_vs_joint_same_n_cs, -manual_vs_joint_max_abs_diff), name)

n,median_max_abs_diff,max_max_abs_diff,same_n_cs
<int>,<dbl>,<dbl>,<dbl>
3600,1.451812e-10,0.0009354537,1


comparison,n,no_cs,too_few_cs_var,with_cor
<chr>,<int>,<int>,<int>,<int>
estimated,3600,0,1429,2171
true_weight,3600,0,1431,2169


## Summary by setting

In [11]:
q <- function(x, p) if (all(is.na(x))) NA_real_ else unname(quantile(x, p, na.rm = TRUE))
summ <- agree %>%
  group_by(comparison, comparison_label, ncp, causal, prop) %>%
  summarise(n_sim = n(), n_with_cor = sum(!is.na(pip_cor)),
            median_cor = q(pip_cor, .5), q25_cor = q(pip_cor, .25), q75_cor = q(pip_cor, .75),
            median_n_cs_var = median(n_cs_var),
            median_max_abs_diff_cs = median(max_abs_diff_cs, na.rm = TRUE),
            same_n_cs = mean(n_cs_insample == n_cs_ref), .groups = "drop")
save_data(summ, paste0(name, "_Summary"))

## by NCP, over causal counts and shares
agree %>% group_by(comparison_label, ncp) %>%
  summarise(n_with_cor = sum(!is.na(pip_cor)), median_cor = round(q(pip_cor, .5), 3),
            q25 = round(q(pip_cor, .25), 3), median_max_abs_diff_cs = round(median(max_abs_diff_cs, na.rm = TRUE), 3),
            same_n_cs = round(mean(n_cs_insample == n_cs_ref), 3), .groups = "drop") %>%
  as.data.frame()

## by ADSP share
agree %>% group_by(comparison_label, prop) %>%
  summarise(n_with_cor = sum(!is.na(pip_cor)), median_cor = round(q(pip_cor, .5), 3),
            q25 = round(q(pip_cor, .25), 3), same_n_cs = round(mean(n_cs_insample == n_cs_ref), 3), .groups = "drop") %>%
  as.data.frame()

## lowest per-setting median correlation of each fit (for the text)
summ %>% group_by(comparison_label) %>% slice_min(median_cor, n = 1, with_ties = FALSE) %>%
  select(comparison_label, ncp, causal, prop, n_with_cor, median_cor, q25_cor) %>% as.data.frame()

comparison_label,ncp,n_with_cor,median_cor,q25,median_max_abs_diff_cs,same_n_cs
<fct>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Estimated mixture,80,622,1.000,0.998,0.009,0.999
Estimated mixture,120,582,0.999,0.998,0.010,0.999
Estimated mixture,250,522,0.999,0.995,0.013,0.996
Estimated mixture,500,445,0.997,0.990,0.013,0.991
True-weight mixture,80,622,1.000,0.998,0.009,0.999
True-weight mixture,120,581,0.999,0.998,0.010,0.999
True-weight mixture,250,520,0.999,0.995,0.013,0.997
True-weight mixture,500,446,0.997,0.989,0.013,0.991


comparison_label,prop,n_with_cor,median_cor,q25,same_n_cs
<fct>,<chr>,<int>,<dbl>,<dbl>,<dbl>
Estimated mixture,0.20,731,0.999,0.996,0.995
Estimated mixture,0.50,722,0.999,0.996,0.996
Estimated mixture,0.80,718,0.999,0.997,0.998
True-weight mixture,0.20,731,0.999,0.995,0.994
True-weight mixture,0.50,722,0.999,0.996,0.996
True-weight mixture,0.80,716,0.999,0.997,0.999


comparison_label,ncp,causal,prop,n_with_cor,median_cor,q25_cor
<fct>,<dbl>,<int>,<chr>,<int>,<dbl>,<dbl>
Estimated mixture,500,3,0.20,78,0.9903805,0.9775081
True-weight mixture,500,3,0.20,79,0.9896883,0.9758309


## Credible set variants of all sampled simulations (scatter)

One row per variant in a credible set of the in-sample or estimated-mixture fit, over every sampled simulation. Used only for plotting; agreement statistics stay per simulation.

In [12]:
cs_pts %>% group_by(prop) %>%
  summarise(simulations = n_distinct(file), variants = n(), causal_variants = sum(causal_variant), .groups = "drop")
save_data(cs_pts, "FigS10_v2.ADSP_UKB_PIP_CS_Variants")

prop,simulations,variants,causal_variants
<chr>,<int>,<int>,<int>
0.20,1200,13402,2345
0.50,1200,12182,2345
0.80,1200,12662,2335
